## Import lib

In [39]:

import pandas as pd 
import numpy as np 
import os
import sys
sys.path.append('../')
sys.path.append('../..')
import random

from sklearn.metrics import classification_report, confusion_matrix \
        , roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from preprocessing.load_data import DataLoader

In [40]:
from detection.isolation_forest import IsolationForestDetector
from detection.lof import LocalOutlierFactorDetector
from detection.autoencoder import AutoencoderDetector

## Read data

In [41]:
training_data_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\train'

test_data_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\test'
test_labels_path = '..\\..\\data\\raw\\SMD_data_use\\ServerMachineDataset\\test_label'

Link tham khao 1 bai anomaly voi bo data set nay: https://www.kaggle.com/code/imenbenamar1/anomalydetection-multivariate-time-series   

Trong code có cả cell đặt tên cho cột  

In [42]:
new_column_names = [
    'cpu_r', 'load_1', 'load_5', 'load_15', 'mem_shmem', 'mem_u', 'mem_u_e', 'total_mem',
    'disk_q', 'disk_r', 'disk_rb', 'disk_svc', 'disk_u', 'disk_w', 'disk_wa', 'disk_wb',
    'si', 'so', 'eth1_fi', 'eth1_fo', 'eth1_pi', 'eth1_po', 'tcp_tw', 'tcp_use',
    'active_opens', 'curr_estab', 'in_errs', 'in_segs', 'listen_overflows', 'out_rsts',
    'out_segs', 'passive_opens', 'retransegs', 'tcp_timeouts', 'udp_in_dg', 'udp_out_dg',
    'udp_rcv_buf_errs', 'udp_snd_buf_errs'
]

In [52]:
training_data = pd.read_csv(training_data_path + '\\machine-1-1.txt', header=None)
print("Training data shape: ", training_data.shape)

training_data.head(2)

Training data shape:  (28479, 38)


,0,1,2,3,4,5,6,7,8,9,...,28,29,30,31,32,33,34,35,36,37
0,0.032258,0.039195,0.027871,0.024390,0.0,0.915385,0.343691,0.0,0.020011,0.000122,...,0.0,0.004298,0.029993,0.022131,0.0,0.000045,0.034677,0.034747,0.0,0.0
1,0.043011,0.048729,0.033445,0.025552,0.0,0.915385,0.344633,0.0,0.019160,0.001722,...,0.0,0.004298,0.030041,0.028821,0.0,0.000045,0.035763,0.035833,0.0,0.0


In [ ]:
training_data.columns = new_column_names

training_data

,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
0,0.032258,0.039195,0.027871,0.024390,0.0,0.915385,0.343691,0.0,0.020011,0.000122,...,0.0,0.004298,0.029993,0.022131,0.000000,0.000045,0.034677,0.034747,0.0,0.0
1,0.043011,0.048729,0.033445,0.025552,0.0,0.915385,0.344633,0.0,0.019160,0.001722,...,0.0,0.004298,0.030041,0.028821,0.000000,0.000045,0.035763,0.035833,0.0,0.0
2,0.043011,0.034958,0.032330,0.025552,0.0,0.915385,0.344633,0.0,0.020011,0.000122,...,0.0,0.004298,0.026248,0.021101,0.000000,0.000045,0.033012,0.033082,0.0,0.0
3,0.032258,0.028602,0.030100,0.024390,0.0,0.912821,0.342750,0.0,0.021289,0.000000,...,0.0,0.004298,0.030169,0.025733,0.000000,0.000022,0.035112,0.035182,0.0,0.0
4,0.032258,0.019068,0.026756,0.023229,0.0,0.912821,0.342750,0.0,0.018734,0.000000,...,0.0,0.004298,0.027240,0.022645,0.000000,0.000034,0.033447,0.033517,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28474,0.075269,0.046610,0.071349,0.076655,0.0,0.928205,0.269303,0.0,0.031649,0.000244,...,0.0,0.008596,0.068980,0.049408,0.000386,0.000034,0.064504,0.064572,0.0,0.0
28475,0.086022,0.070975,0.075808,0.077816,0.0,0.930769,0.269303,0.0,0.029946,0.000244,...,0.0,0.008596,0.073029,0.055584,0.000386,0.000034,0.067690,0.067757,0.0,0.0
28476,0.086022,0.065678,0.073579,0.076655,0.0,0.935897,0.270245,0.0,0.030372,0.000244,...,0.0,0.008596,0.070516,0.048893,0.000386,0.000034,0.064866,0.064934,0.0,0.0
28477,0.086022,0.056144,0.068004,0.074332,0.0,0.933333,0.271186,0.0,0.032643,0.000244,...,0.0,0.008596,0.070308,0.055069,0.000386,0.000045,0.067111,0.067178,0.0,0.0


In [44]:
test_data = pd.read_csv(test_data_path + '\\machine-1-1.txt', header=None)
test_data.columns = new_column_names
print(test_data.shape)
test_data.head(2)

(28479, 38)


,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,...,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs
0,0.075269,0.065678,0.070234,0.074332,0.0,0.933333,0.274011,0.0,0.031081,0.000000,...,0.0,0.008596,0.068036,0.048893,0.000386,0.000034,0.064432,0.064500,0.0,0.0
1,0.086022,0.080508,0.075808,0.076655,0.0,0.930769,0.274953,0.0,0.031081,0.000122,...,0.0,0.008596,0.070020,0.050437,0.000386,0.000022,0.065228,0.065224,0.0,0.0


In [45]:
test_label = pd.read_csv(test_labels_path + '\\machine-1-1.txt', header=None)

print(len(test_label))

28479


## LOF 

In [46]:
lof = LocalOutlierFactorDetector(n_neighbors=20, contamination=0.01, novelty=True)

lof.fit(training_data)

print("Model training completed.")

print("Scoring test data...")

lof_scores = lof.score(test_data)

lof_predictions = lof.predict(test_data)

print("Local Outlier Factor Scores: ", lof_scores[:5])
print("Local Outlier Factor Predictions: ", lof_predictions[:5])

Model training completed.
Scoring test data...


d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(
d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


Local Outlier Factor Scores:  [-0.9878004  -1.05870158 -0.98900554 -1.04249601 -1.06323947]
Local Outlier Factor Predictions:  [False False False False False]


In [55]:

print("Evaluating model performance...")

if test_label is not None:
    print("Number of anomalies detected: ", np.sum(lof_predictions))
    print("Percentage of anomalies detected: ", np.mean(lof_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(test_label, lof_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(test_label, lof_predictions.astype(int)))

    ## vi score cua LOF la am. ví du score < -1 -> anomaly nen can dung -scores
    print("AUC-ROC of local outlier factor: ", roc_auc_score(test_label, -lof_scores))

    if not os.path.exists('../../models/checkpoint_SMD/'):
        os.makedirs('../../models/checkpoint_SMD/')

    result_folder = '../../models/checkpoint_SMD/baseline_lof/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    lof.save_model(result_folder + 'lof.pkl')

    classification_report_df = classification_report(test_label, lof_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'lof_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(lof_predictions))

Evaluating model performance...
Number of anomalies detected:  8440
Percentage of anomalies detected:  29.635872046069032 %
Confusion Matrix:
[[19707  6078]
 [  332  2362]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.76      0.86     25785
           1       0.28      0.88      0.42      2694

    accuracy                           0.77     28479
   macro avg       0.63      0.82      0.64     28479
weighted avg       0.92      0.77      0.82     28479

AUC-ROC of local outlier factor:  0.9006921636126735


## Isolation forest

In [53]:
isolation_forest = IsolationForestDetector(contamination=0.01, random_state=42)

# vi isolation forest la model unsupervised, nen ta fit method tren test data
isolation_forest.fit(training_data)
# isolation_forest.fit(test_data)

isolation_forest_scores = isolation_forest.score(test_data)
isolation_forest_predictions = isolation_forest.predict(test_data)

print("Isolation Forest Scores: ", isolation_forest_scores[:5])
print("Isolation Forest Predictions: ", isolation_forest_predictions[:5])

d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(
d:\thac_si_phenika\master_thesis\Opstimus\venv_opstimus\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(


Isolation Forest Scores:  [-0.21340577 -0.21946956 -0.23198517 -0.20146809 -0.20376946]
Isolation Forest Predictions:  [False False False False False]


In [54]:

print("Evaluating model performance...")

if test_label is not None:
    print("Number of anomalies detected: ", np.sum(isolation_forest_predictions))
    print("Percentage of anomalies detected: ", np.mean(isolation_forest_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(test_label, isolation_forest_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(test_label, isolation_forest_predictions.astype(int)))

    print("AUC-ROC of isolation forest: ", roc_auc_score(test_label, isolation_forest_scores))

    if not os.path.exists('../../models/checkpoint_SMD/'):
        os.makedirs('../../models/checkpoint_SMD/')

    result_folder = '../../models/checkpoint_SMD/baseline_isolation_forest/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    isolation_forest.save_model(result_folder + 'isolation_forest.pkl')

    classification_report_df = classification_report(test_label, isolation_forest_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'isolation_forest_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(isolation_forest_predictions))

Evaluating model performance...
Number of anomalies detected:  3711
Percentage of anomalies detected:  13.030654166227748 %
Confusion Matrix:
[[23177  2608]
 [ 1591  1103]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.90      0.92     25785
           1       0.30      0.41      0.34      2694

    accuracy                           0.85     28479
   macro avg       0.62      0.65      0.63     28479
weighted avg       0.88      0.85      0.86     28479

AUC-ROC of isolation forest:  0.8226006225600049
